In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from core.interface import Model, ready, lift, ensemble, learn_from
from core.label import Label
from core.interpreter import _get_train_df_for_label, _get_test_df_for_label

from dataclasses import dataclass
import polars as pl
import datetime as dt

In [ ]:
@dataclass
class TestModel:
    x_column: str
    value = None
    hyperparameter = None

    def fit(self, training_set: pl.DataFrame):
        self.value = df[self.x_column].mean()

    def predict(self, df: pl.DataFrame):
        hyperparameter = self.hyperparameter or 0
        return [self.value + hyperparameter] * df.shape[0]


In [ ]:
times = pl.datetime_range(
    dt.datetime(2025, 1, 1),
    dt.datetime(2025, 1, 7),
    interval = '1d',
    eager=True
).rename('ts').to_frame()

df = times.with_columns(x = pl.arange(0, times.shape[0]))

### Rolling retrain

In [ ]:
model = ready(
    label='tester',
    model_constructor = lambda: TestModel(x_column='x')
    )

In [ ]:
retrain_points = [
    dt.datetime(2025, 1, 3),
    dt.datetime(2025, 1, 5)
]

rolling_train = lift(
    model=model,
    atomics = retrain_points,
    train_mask_for_label = lambda label: pl.col('ts') <= label,
    test_mask_for_label = lambda label: pl.col('ts') > label,
    name='RollingRetrain'
)

In [ ]:
rolling_train.fit(df)

In [ ]:
rolling_train.predict(df)

### Randomised Cross Validation

In [ ]:
# For this one we need a bigger test dataframe
times = pl.datetime_range(
    dt.datetime(2025, 1, 1),
    dt.datetime(2025, 1, 7),
    interval = '1m',
    eager=True
).rename('ts').to_frame()

df = times.with_columns(x = pl.arange(0, times.shape[0]))

In [ ]:
model = ready(
    label='tester',
    model_constructor= lambda: TestModel(x_column='x')
    )

In [ ]:
def random_assignment_expr(column, num_categories, seed=0) -> pl.Expr:
    return pl.col(column).hash(seed=seed).mod(num_categories)

train_mask_for_label = lambda fold_number: random_assignment_expr('ts', 5) == fold_number

randomised_cv = lift(
    model=model,
    atomics=range(5),
    train_mask_for_label=train_mask_for_label,
    test_mask_for_label = lambda fold_number: ~train_mask_for_label(fold_number),
    name='RandomFold'
)

In [ ]:
# for fold in range(5):
#     label = Label(RandomFold=fold, leaf='tester')
#     train_set_size =  _get_train_df_for_label(randomised_cv.root,  df, label).shape[0]/df.shape[0]
#     test_set_size = _get_test_df_for_label(randomised_cv,  df, label).shape[0]/df.shape[0]
#     print(f'Train Set Size: {train_set_size:.2f}, Test set size {test_set_size:.2f}'
#     )

#### Ensemble two models

In [ ]:
from functools import partial

In [ ]:
@dataclass
class TestModel:
    x_column: str
    value = None
    hyperparameter = None

    def fit(self, df: pl.DataFrame):
        self.value = df[self.x_column].mean()

    def predict(self, df: pl.DataFrame):
        hyperparameter = self.hyperparameter or 0
        return [self.value + hyperparameter] * df.shape[0]


In [ ]:
model_1 = partial(TestModel, hyperparameter=1)
model_2 = partial(TestModel, hyperparameter=2)

In [ ]:
ens = ensemble(
    'ensemble',
    ready(model_1, 'model_1'),
    ready(model_2, 'model_2')
)

In [ ]:
ens.print_tree()

In [ ]:
for c in ens.collect_labels():
    print(c)